# Creating a model from scratch

**This notebook is under construction. Please do not refer to it.**

In [1]:
import pycellin as pc
import pycellin.graph.properties as pc_core

In [ ]:
lin1 = pc.CellLineage()  # division
lin1.add_nodes_from(
    [
        (1, {"frame": 0, "cell_ID": 1, "lineage_ID": 1, "cell_x": 10, "cell_y": 0}),
        (2, {"frame": 1, "cell_ID": 2, "lineage_ID": 1, "cell_x": 20, "cell_y": 0}),
        (3, {"frame": 2, "cell_ID": 3, "lineage_ID": 1, "cell_x": 30, "cell_y": 0}),
        (4, {"frame": 2, "cell_ID": 4, "lineage_ID": 1, "cell_x": 40, "cell_y": 0}),
        (5, {"frame": 3, "cell_ID": 5, "lineage_ID": 1, "cell_x": 50, "cell_y": 0}),
        (6, {"frame": 4, "cell_ID": 6, "lineage_ID": 1, "cell_x": 60, "cell_y": 0}),
    ]
)
lin1.add_edges_from([(1, 2), (2, 3), (2, 4), (3, 5), (5, 6)])
lin1.graph["lineage_ID"] = 1

lin2 = pc.CellLineage()  # gap
lin2.add_nodes_from(
    [
        (7, {"frame": 0, "cell_ID": 7, "lineage_ID": 2, "cell_x": 0, "cell_y": 10}),
        (8, {"frame": 1, "cell_ID": 8, "lineage_ID": 2, "cell_x": 0, "cell_y": 20}),
        (9, {"frame": 3, "cell_ID": 9, "lineage_ID": 2, "cell_x": 0, "cell_y": 30}),
    ]
)
lin2.add_edges_from([(7, 8), (8, 9)])
lin2.graph["lineage_ID"] = 2

lin3 = pc.CellLineage()  # single-node lineage
lin3.add_node(10, frame=0, cell_ID=10, lineage_ID=3, cell_x=0, cell_y=0)
lin3.graph["lineage_ID"] = 3

In [3]:
model = pc.Model(
    data=pc.Data({1: lin1, 2: lin2, 3: lin3}),
    props_metadata=pc.PropsMetadata(
        {
            "frame": pc_core.create_frame_property(),
            "cell_x": pc_core.create_cell_coord_property(axis="x", unit="pixel"),
            "cell_y": pc_core.create_cell_coord_property(axis="y", unit="pixel"),
            "cell_ID": pc_core.create_cell_id_property(),
            "lineage_ID": pc_core.create_lineage_id_property(),
        }
    ),
    reference_time_property="frame",
)
print(model)

Model with 3 lineages.


## Adding metadata

In [4]:
model.model_metadata.get_standard_metadata()

{'reference_time_property': 'frame',
 'time_step': 1,
 'time_unit': None,
 'pixel_width': None,
 'pixel_height': None,
 'pixel_depth': None,
 'space_unit': None,
 'pycellin_version': '0.4.2',
 'creation_timestamp': '2026-06-11T15:48:50.152072',
 'name': None,
 'provenance': None,
 'file_location': None}

In [5]:
model.model_metadata.name = "test_model_from_scratch"
model.model_metadata.provenance = "LX"
model.model_metadata.time_unit = "frame"
model.model_metadata.pixel_width = 1
model.model_metadata.pixel_height = 1
model.model_metadata.space_unit = "pixel"

In [6]:
model.model_metadata.get_standard_metadata()

{'reference_time_property': 'frame',
 'time_step': 1,
 'time_unit': 'frame',
 'pixel_width': 1,
 'pixel_height': 1,
 'pixel_depth': None,
 'space_unit': 'pixel',
 'pycellin_version': '0.4.2',
 'creation_timestamp': '2026-06-11T15:48:50.152072',
 'name': 'test_model_from_scratch',
 'provenance': 'LX',
 'file_location': None}

Important, can add whatever you want.  
*give example and stress the importance of putting a lot of info*

In [7]:
model.model_metadata.segmentation = "ilastik pixel classifier"
model.model_metadata.tracking = "manual"
model.model_metadata.objective = "100X oil"

In [8]:
model.model_metadata.get_custom_metadata()

{'segmentation': 'ilastik pixel classifier',
 'tracking': 'manual',
 'objective': '100X oil'}

## Constructing lineages

In [8]:
new_lin_id = model.add_lineage()
new_lin_id

1

In [9]:
print(model)

Model named 'test_model_from_scratch' with 1 lineage, built from LX.


In [10]:
model.get_cell_lineage_IDs()

[1]

In [11]:
model.add_cell(lid=1, cid=1, time_value=0)
model.add_cell(lid=1, cid=2, time_value=1)
model.add_cell(lid=1, cid=3, time_value=2)
model.add_cell(lid=1, cid=4, time_value=2)
model.add_cell(lid=1, cid=5, time_value=3)

model.add_link(source_cid=1, target_cid=2, source_lid=1)
model.add_link(source_cid=2, target_cid=3, source_lid=1)
model.add_link(source_cid=2, target_cid=4, source_lid=1)
model.add_link(source_cid=3, target_cid=5, source_lid=1)

In [12]:
model.set_time_step()
model.get_time_step()

1

In [9]:
model.get_node_properties()

{'timepoint': Property(identifier='timepoint', name='Timepoint', description='Timepoint of the detection', provenance='pycellin', prop_type=<PropertyType.NODE: 1>, lin_type='CellLineage', dtype='int', unit=None)}

In [13]:
new_lin = model.get_cell_lineage_from_ID(1)
new_lin.plot()

KeyError: 'Attribute does not exist'

## Adding properties

In [9]:
pc.export_TrackMate_XML(
    model,
    "/media/lxenard/data/Code/pycellin/pycellin/sample_data/results/test_model_from_scratch.xml",
    {"space": "pixel", "time": "frame"},
)

Model(model_metadata=ModelMetadata(reference_time_property='frame', time_step=1, time_unit='frame', pixel_width=1, pixel_height=1, pixel_depth=None, space_unit='pixel', pycellin_version='0.4.2', creation_timestamp='2026-06-11T15:48:50.152072', name='test_model_from_scratch', provenance='LX', file_location=None), props_metadata=PropsMetadata(props={'RADIUS': Property(identifier='RADIUS', name='Radius', description='Radius', provenance='TrackMate', prop_type=<PropertyType.NODE: 1>, lin_type='CellLineage', dtype='float', unit='pixel'), 'TRACK_ID': Property(identifier='TRACK_ID', name='Lineage ID', description='Track ID', provenance='pycellin', prop_type=<PropertyType.LINEAGE: 4>, lin_type='Lineage', dtype='int', unit=None), 'FRAME': Property(identifier='FRAME', name='Frame', description='Frame number of the cell', provenance='pycellin', prop_type=<PropertyType.NODE: 1>, lin_type='CellLineage', dtype='int', unit='frame'), 'SPOT_SOURCE_ID': Property(identifier='SPOT_SOURCE_ID', name='SPOT_S